# DeepSketch - Sample Training (E2E Vector Style)
This notebook orchestrates end-to-end training on the 500-image sample batch.
It uses the pre-split Kaggle dataset and links the GitHub repository for training execution.

## 1. Check Kaggle GPU & Setup Environment

In [ ]:
import os
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the GitHub Repository

In [ ]:
!rm -rf stylized-portrait-generation
!git clone https://github.com/supratimcoder1/stylized-portrait-generation.git
%cd stylized-portrait-generation

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt -q

## 4. Verify Dataset Location

In [ ]:
import os

candidates = ["/kaggle/input/sample-feret/sample_dataset", "/kaggle/input/sample-feret"]
splits = ["train_images", "train_targets", "val_images", "val_targets"]
base_dir = None

for candidate in candidates:
    if all(os.path.exists(os.path.join(candidate, s)) for s in splits):
        base_dir = candidate
        break

if base_dir is None:
    raise FileNotFoundError("Could not find split folders under /kaggle/input/sample-feret")

print(f"Using dataset base: {base_dir}")
for split in splits:
    path = os.path.join(base_dir, split)
    print(f"{split} found: {len(os.listdir(path))} files")

## 5. Create Dataset Symlink
Link the Kaggle input dataset into the repo's expected dataset/ location.

In [ ]:
!rm -rf dataset
!mkdir -p dataset
!ln -s "$base_dir/train_images" dataset/train_images
!ln -s "$base_dir/train_targets" dataset/train_targets
!ln -s "$base_dir/val_images" dataset/val_images
!ln -s "$base_dir/val_targets" dataset/val_targets

!ls -la dataset/

## 6. Start Training

In [ ]:
!python training/train.py --epochs 100 --batch-size 8 --output-dir /kaggle/working/training_outputs --checkpoint-dir /kaggle/working/checkpoints

## 7. Inspect Training Outputs

In [ ]:
import glob
from IPython.display import display, Image as IPImage

sample_files = sorted(glob.glob("/kaggle/working/training_outputs/epoch_*.png"))
if sample_files:
    latest = sample_files[-1]
    print(f"Showing: {latest}")
    display(IPImage(filename=latest, width=800))
else:
    print("No sample images found - training may not have completed.")

## 8. Download Best Model

In [ ]:
!zip -j /kaggle/working/best_generator.zip /kaggle/working/checkpoints/best_generator.pth
print("Best model zipped to /kaggle/working/best_generator.zip")